**MINOR PROJECT**

---


COMBINED 2 DATASETS

# **DEPENDENCIES**

In [1]:
pip install  pandas numpy scikit-learn boto3 joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 6.6 MB/s eta 0:00:00


# **AWS_S3 AND LOG**

In [2]:
#Dependencies
import os
import boto3
from google.colab import userdata
import logging #various stages of the process for better debugging.

# AWS credentials from Colab secrets
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('Secret_access_key')
os.environ['AWS_DEFAULT_REGION'] = 'ap-south-1'

s3_client = boto3.client('s3')
bucket_name = 'sandipproject'

# Logging setup
log_path = '/content/model_training.log'
logging.basicConfig(filename=log_path, level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Environment and AWS credentials set up")

# **Load & Clean Datasets**
Combined dataset with 1337 rows by duplicating Diabetes data.

In [3]:
#Dependencies
import pandas as pd #for handling data operations.
import numpy as np #for array operations.
from sklearn import datasets
from sklearn.model_selection import train_test_split #splitting the dataset.
from botocore.exceptions import ClientError #handle AWS_S3 errors.

# Paths for dataset files
X_combined_path = '/content/X_combined.csv'
y_combined_path = '/content/y_combined.csv'

# Regenerate dataset
logging.info("Regenerating combined dataset to ensure 1337 rows.")
print("Regenerating combined dataset to ensure 1337 rows.")
breast_cancer = datasets.load_breast_cancer()
diabetes = datasets.load_diabetes()
X_bc = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
y_bc = pd.Series(breast_cancer.target)
X_diabetes = pd.DataFrame(diabetes.data, columns=[f"feature_{i}" for i in range(diabetes.data.shape[1])])
y_diabetes = pd.Series((diabetes.target > diabetes.target.mean()).astype(int))

# Duplicate Diabetes data to reach 768 samples (569 + 768 = 1337)
X_diabetes_extended = pd.concat([X_diabetes, X_diabetes.sample(n=768 - X_diabetes.shape[0], replace=True, random_state=42)], ignore_index=True)
y_diabetes_extended = pd.concat([y_diabetes, y_diabetes.sample(n=768 - y_diabetes.shape[0], replace=True, random_state=42)], ignore_index=True)

# Pad Diabetes data to 30 columns
X_diabetes_padded = np.pad(X_diabetes_extended.values, ((0, 0), (0, 30 - X_diabetes_extended.shape[1])), mode='constant', constant_values=0)
X_diabetes_padded = pd.DataFrame(X_diabetes_padded, columns=X_bc.columns)

# Combine datasets
X_combined = pd.concat([X_bc, X_diabetes_padded], ignore_index=True)
y_combined = pd.concat([y_bc, y_diabetes_extended], ignore_index=True)

#Shapes
print(f"X_bc shape: {X_bc.shape}")
print(f"X_diabetes_padded shape: {X_diabetes_padded.shape}")
print(f"y_bc shape: {y_bc.shape}")
print(f"y_diabetes_extended shape: {y_diabetes_extended.shape}")
print(f"X_combined shape before saving: {X_combined.shape}")
print(f"y_combined shape before saving: {y_combined.shape}")

# Convert y_combined to DataFrame with 'target' column
y_combined_df = pd.DataFrame(y_combined, columns=['target'])

# Save to local files and upload to S3
X_combined.to_csv(X_combined_path, index=False)
y_combined_df.to_csv(y_combined_path, index=False)
s3_client.upload_file(X_combined_path, bucket_name, 'X_combined.csv')
s3_client.upload_file(y_combined_path, bucket_name, 'y_combined.csv')
logging.info("Generated and uploaded corrected combined dataset to S3 (1337 rows)")
print("Generated and uploaded corrected dataset to S3")

# Retrieve the dataset from S3
try:
    s3_client.download_file(bucket_name, 'X_combined.csv', X_combined_path)
    s3_client.download_file(bucket_name, 'y_combined.csv', y_combined_path)
    X_combined = pd.read_csv(X_combined_path)
    y_combined = pd.read_csv(y_combined_path)['target']
    logging.info("Combined dataset retrieved from S3")
    print("Dataset successfully retrieved from S3")
except ClientError as e:
    logging.error(f"Failed to retrieve dataset from S3: {e}")
    print(f"Error retrieving dataset from S3: {e}")
    raise

# Verify shapes
print(f"Raw combined dataset shape: {X_combined.shape}")
print(f"Raw combined labels shape: {y_combined.shape}")

# Clean dataset
X_combined = X_combined.fillna(0)
y_combined = y_combined[X_combined.index]
print(f"Combined dataset shape after cleaning: {X_combined.shape}")
print(f"Combined labels shape after cleaning: {y_combined.shape}")

# Split for training and testing
X_train, X_test, y_train, Y_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42) #training 80% testing 20%./ reproducibility of random partitions.
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Regenerating combined dataset to ensure 1337 rows.
X_bc shape: (569, 30)
X_diabetes_padded shape: (768, 30)
y_bc shape: (569,)
y_diabetes_extended shape: (768,)
X_combined shape before saving: (1337, 30)
y_combined shape before saving: (1337,)
Generated and uploaded corrected dataset to S3
Dataset successfully retrieved from S3
Raw combined dataset shape: (1337, 30)
Raw combined labels shape: (1337,)
Combined dataset shape after cleaning: (1337, 30)
Combined labels shape after cleaning: (1337,)
Training set shape: (1069, 30)
Test set shape: (268, 30)


# **File Integrity Functions with S3 and Timestamping**
---


File hashing, store and verify dataset integrity in S3, and retrieve for
training.


In [4]:
# Dependencies
import hashlib  # Used for hashing files to ensure integrity
import pandas as pd  #data handling
from datetime import datetime  # timestamping
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type  # retrying operations
import boto3  # AWS S3 interactions
import logging  # logging actions and errors
from sklearn.preprocessing import StandardScaler  # Standardizes data for ML
from sklearn.model_selection import train_test_split  # splits dataset into train/test sets
import os  # file path operations
import numpy as np  #dummy data generation

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# AWS S3 client setup.
s3_client = boto3.client('s3')
bucket_name = 'sandipproject'  # S3 bucket name

# Sample combined dataset
X_combined = pd.DataFrame(np.random.rand(1337, 10), columns=[f'Feature_{i}' for i in range(10)])
y_combined = pd.Series(np.random.randint(0, 2, 1337), name='target')

# File Integrity Verification Using SHA-256
def calculate_file_hash(file_path):
    """Calculate SHA-256 hash of a file."""
    try:
        hasher = hashlib.sha256()
        with open(file_path, 'rb') as f:
            hasher.update(f.read())
        return hasher.hexdigest()
    except FileNotFoundError:
        logging.error(f"File {file_path} not found!")
        raise
    except Exception as e:
        logging.error(f"Error hashing {file_path}: {str(e)}")
        raise

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=10),
    retry=retry_if_exception_type(Exception)  # Retry on all exceptions except those we explicitly handle
)
def store_file_hash(file_hash, s3_key):
    """Store file hash with timestamp in S3."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    hash_file_local = '/content/file_hashes.txt'
    # Download existing hashes (if they exist)
    existing_hashes = ""
    try:
        s3_client.download_file(bucket_name, 'file_hashes.txt', hash_file_local)
        with open(hash_file_local, 'r') as f:
            existing_hashes = f.read()
        logging.info("Existing hashes downloaded from S3.")
    except s3_client.exceptions.NoSuchKey:
        logging.info("No existing file_hashes.txt found in S3; creating a new one.")
        existing_hashes = ""
    except s3_client.exceptions.ClientError as e:
        error_code = e.response.get('Error', {}).get('Code')
        if error_code == '404':
            logging.info("No existing file_hashes.txt found in S3 (ClientError 404); creating a new one.")
            existing_hashes = ""
        else:
            logging.error(f"Error downloading existing hashes from S3: {str(e)}")
            raise  # Retry on other S3 errors
    except Exception as e:
        logging.error(f"Error downloading existing hashes from S3: {str(e)}")
        raise  # Retry on other errors

    # Append new hash with timestamp
    try:
        with open(hash_file_local, 'w') as f:
            f.write(existing_hashes + f"{s3_key}: {file_hash} | {timestamp}\n")
        logging.info(f"New hash appended locally: {s3_key}: {file_hash} at {timestamp}")
    except Exception as e:
        logging.error(f"Error writing to local hash file: {str(e)}")
        raise

    # Upload the updated hash file to S3
    try:
        s3_client.upload_file(hash_file_local, bucket_name, 'file_hashes.txt')
        logging.info(f"Stored hash for {s3_key}: {file_hash} at {timestamp} in S3")
    except s3_client.exceptions.ClientError as e:
        error_code = e.response.get('Error', {}).get('Code')
        if error_code == '404':
            logging.warning("Unexpected 404 error during upload; proceeding as file will be created.")
            # Attempt to force upload even if 404 occurs
            s3_client.upload_file(hash_file_local, bucket_name, 'file_hashes.txt')
            logging.info(f"Forced upload of hash file after 404: {s3_key}: {file_hash} at {timestamp}")
        else:
            logging.error(f"Error uploading hash file to S3: {str(e)}")
            raise  # Retry on other S3 errors
    except Exception as e:
        logging.error(f"Error uploading hash file to S3: {str(e)}")
        raise

def check_file_hash(s3_key):
    """Verify file integrity by comparing hashes from S3."""
    local_file = '/content/temp_file'
    try:
        # Download the file from S3 and calculate its hash
        s3_client.download_file(bucket_name, s3_key, local_file)
        file_hash = calculate_file_hash(local_file)
        # Download the hash file
        hash_file_local = '/content/file_hashes.txt'
        try:
            s3_client.download_file(bucket_name, 'file_hashes.txt', hash_file_local)
        except s3_client.exceptions.NoSuchKey:
            logging.warning("No file_hashes.txt found in S3!")
            return False
        except s3_client.exceptions.ClientError as e:
            error_code = e.response.get('Error', {}).get('Code')
            if error_code == '404':
                logging.warning("No file_hashes.txt found in S3 (ClientError 404)!")
                return False
            else:
                logging.error(f"Error downloading hash file from S3: {str(e)}")
                raise
        with open(hash_file_local, 'r') as f:
            stored_hashes = f.readlines()
            for line in stored_hashes:
                stored_key, rest = line.strip().split(": ", 1)
                stored_hash, timestamp = rest.split(" | ")
                if stored_key == s3_key and stored_hash == file_hash:
                    logging.info(f"File {s3_key} integrity verified from S3! Timestamp: {timestamp}")
                    return True
        logging.warning(f"File {s3_key} hash does not match any stored hash!")
        return False
    except s3_client.exceptions.NoSuchKey:
        logging.warning(f"S3 hash file or target file {s3_key} not found!")
        return False
    except s3_client.exceptions.ClientError as e:
        error_code = e.response.get('Error', {}).get('Code')
        if error_code == '404':
            logging.warning(f"S3 hash file or target file {s3_key} not found (ClientError 404)!")
            return False
        else:
            logging.error(f"Error checking hash for {s3_key} from S3: {str(e)}")
            raise
    except Exception as e:
        logging.error(f"Error checking hash for {s3_key} from S3: {str(e)}")
        return False

# Main Workflow: Save Dataset, Verify Integrity, and Preprocess
try:
    # Step 1: Combine and Save Dataset to Local File
    data = pd.concat([X_combined, y_combined.rename("target")], axis=1)
    csv_local_path = '/content/dataset_cleaned.csv'
    data.to_csv(csv_local_path, index=False)
    logging.info(f"Dataset saved locally at {csv_local_path}")

    # Step 2: Upload Dataset to S3
    s3_key = 'dataset_cleaned.csv'
    s3_client.upload_file(csv_local_path, bucket_name, s3_key)
    logging.info(f"Dataset uploaded to S3 at s3://{bucket_name}/{s3_key}")

    # Step 3: Calculate and Store File Hash
    file_hash = calculate_file_hash(csv_local_path)
    store_file_hash(file_hash, s3_key)

    # Step 4: Verify Integrity of the Uploaded Dataset
    if check_file_hash(s3_key):
        logging.info("Cleaned dataset file integrity is valid!")
        print("Cleaned dataset file integrity is valid!")
    else:
        logging.warning("Cleaned dataset file integrity check failed!")
        raise Exception("Dataset integrity check failed; halting execution.")

    # Step 5: Retrieve Dataset from S3 for Training
    s3_download_path = '/content/downloaded_dataset.csv'
    s3_client.download_file(bucket_name, s3_key, s3_download_path)
    logging.info(f"Dataset downloaded from S3 to {s3_download_path}")

    # Step 6: Verify Integrity After Download and Preprocess
    if check_file_hash(s3_key):
        data = pd.read_csv(s3_download_path)
        X_cleaned = data.drop(columns='target')
        y_cleaned = data['target']

        # Step 7: Train-Test Split and Standardization
        X_train, X_test, Y_train, Y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)
        scaler = StandardScaler()
        X_train_std = scaler.fit_transform(X_train)
        X_test_std = scaler.transform(X_test)

        logging.info("Dataset retrieved from S3 and preprocessed successfully.")
        print("Dataset retrieved from S3 and preprocessed successfully.")
        print(f"X_train_std shape: {X_train_std.shape}")
        print(f"X_test_std shape: {X_test_std.shape}")
        print(f"Y_train shape: {Y_train.shape}")
        print(f"Y_test shape: {Y_test.shape}")
    else:
        logging.error("Downloaded dataset integrity check failed!")
        raise Exception("Downloaded dataset integrity check failed.")

except Exception as e:
    logging.error(f"Pipeline failed: {str(e)}")
    print(f"Error: {str(e)}")
    raise

# Step 8: Save Preprocessed Data for Future Use (Optional)
preprocessed_data = {
    'X_train_std': X_train_std,
    'X_test_std': X_test_std,
    'Y_train': Y_train,
    'Y_test': Y_test
}
preprocessed_path = '/content/preprocessed_data.pkl'
pd.to_pickle(preprocessed_data, preprocessed_path)
s3_client.upload_file(preprocessed_path, bucket_name, 'preprocessed_data.pkl')
logging.info(f"Preprocessed data saved to S3 at s3://{bucket_name}/preprocessed_data.pkl")
print(f"Preprocessed data saved to S3 at s3://{bucket_name}/preprocessed_data.pkl")

Cleaned dataset file integrity is valid!
Dataset retrieved from S3 and preprocessed successfully.
X_train_std shape: (1069, 10)
X_test_std shape: (268, 10)
Y_train shape: (1069,)
Y_test shape: (268,)
Preprocessed data saved to S3 at s3://sandipproject/preprocessed_data.pkl


# **ANOMALY**
Anomaly detection, or outlier detection, is the identification of observations, events or data points that deviate from what is usual, standard or expected, making them inconsistent with the rest of a data set.

The goal is to identify unusual data points in the test set using IsolationForest.

**Isolation Forest**


1. Speed - Fast, even with big datasets.
2. Memory -  Use Low.
3. Big Data - Can handle large amounts.
4. Setup - Simple, few lines of code.

In [5]:
#Dependencies
import numpy as np
import matplotlib.pyplot as plt #For plotting anomaly visualizations.
import joblib # Used to save the trained model to disk.
from sklearn.ensemble import IsolationForest

# Anomaly detection using Isolation Forest
def detect_anomalies(X_data_std):
    """Detect anomalies using Isolation Forest and return anomalies and model."""
    isolation_forest = IsolationForest(contamination=0.1, random_state=42, n_jobs=-1) #assuming 10% of the data will be anomalies./ Uses (n_jobs=-1) all available CPU cores for faster computation.
    anomalies = isolation_forest.fit_predict(X_data_std)
    return anomalies, isolation_forest # return both anomalies and the model(I_F)

# Train the model and get anomalies on test set
anomalies, isolation_forest = detect_anomalies(X_test_std)  # Use X_test_std
n_anomalies = np.sum(anomalies == -1)
logging.info(f"Anomaly Detection: {n_anomalies} anomalies detected in the test set.")
print(f"Number of anomalies detected: {n_anomalies}")

# Visualize anomalies (using first two features for 2D plot)
plt.figure(figsize=(8, 6))
plt.scatter(X_test_std[:, 0], X_test_std[:, 1], c=anomalies, cmap='coolwarm', alpha=0.6)
plt.title("Anomalies in Test Data (Features 0 and 1)")
plt.xlabel("Feature 0 (Standardized)")
plt.ylabel("Feature 1 (Standardized)")
plt.colorbar(label='Anomaly (-1) or Normal (1)')
anomaly_plot_path = '/content/anomaly_plot.png'
plt.savefig(anomaly_plot_path)
plt.close()
s3_client.upload_file(anomaly_plot_path, bucket_name, 'anomaly_plot.png')
logging.info(f"Anomaly plot uploaded to S3 at s3://{bucket_name}/anomaly_plot.png")
print("Anomaly plot saved and uploaded to S3")

# Save the IsolationForest model to S3
isolation_forest_path = '/content/isolation_forest.pkl'
joblib.dump(isolation_forest, isolation_forest_path)
s3_client.upload_file(isolation_forest_path, bucket_name, 'isolation_forest.pkl')
logging.info(f"IsolationForest model saved to S3 at s3://{bucket_name}/isolation_forest.pkl")
print("Isolation Forest model saved and uploaded to S3")

Number of anomalies detected: 27
Anomaly plot saved and uploaded to S3
Isolation Forest model saved and uploaded to S3


# **Model Training with Hyperparameter Tuning**


---


Train a neural network with hyperparameter tuning and save the best model.
* 0.001 is a standard learning rate for Adam, balancing speed and stability.
* 64 units are a moderate size for my dataset, avoiding underfitting or excessive complexity.
* 0.3 dropout rate is a common choice to prevent overfitting without overly weakening the model. ( Drops 30% of neurons during training to reduce overfitting )


---


--ReLU (Rectified Linear Unit) activation introduces non-linearity, enabling the network to learn complex patterns from 30 features. It’s computationally efficient and widely used. f(x) = max(0, x). This means it returns
the input value if it's positive, and zero if it's negative.

---




--The Adam optimizer is chosen for anomaly detection due to its adaptive learning rate and momentum, which efficiently handle the complex, non-linear patterns in my combined dataset (Breast Cancer + Diabetes), ensuring stable and fast convergence.


---

sparse_categorical_crossentropy: Optimized loss function for integer labels (0 or 1) from Y_train, ideal for multi-class classification with efficient computation.

In [6]:
# Dependencies
import tensorflow as tf  # TensorFlow for neural networks
from tensorflow.keras.models import Sequential  # Sequential for model building
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout  # Dense layers, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam  # Adam optimizer
import numpy as np  #dummy data generation (if needed)
import pandas as pd  #data handling (if needed)
import joblib  # Joblib for scaler saving
import logging  #logging actions and errors
import boto3  #AWS S3 interactions

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# AWS S3 client setup.
s3_client = boto3.client('s3')
bucket_name = 'sandipproject'  # S3 bucket name

# Verify dataset shape
try:
    print(f"X_train_std shape: {X_train_std.shape}")
    print(f"Y_train shape: {Y_train.shape}")
except NameError:
    raise Exception("X_train_std and Y_train must be defined in Colab environment. Please ensure they are loaded from prior steps.")

# Model creation function (updated to dynamically set input_shape)
def create_model(learning_rate=0.001, units=64, dropout_rate=0.3):
    """Create a neural network model with specified hyperparameters."""
    input_dim = X_train_std.shape[1]  # Dynamically set input shape based on dataset
    model = Sequential([
        Dense(units, activation='relu', input_shape=(input_dim,)),  # Updated to use input_dim
        BatchNormalization(),
        Dense(units // 2, activation='relu'),
        Dropout(dropout_rate),
        Dense(2, activation='softmax')  # Add output layer for binary classification
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Hyperparameter tuning grid search
learning_rates = [0.001, 0.01]
units_options = [32, 64]
batch_sizes = [16, 32]
epochs_options = [10, 20]
best_model = None
best_accuracy = 0
history = None

for lr in learning_rates:
    for units in units_options:
        for batch_size in batch_sizes:
            for epochs in epochs_options:
                model = create_model(learning_rate=lr, units=units)
                history_temp = model.fit(X_train_std, Y_train, epochs=epochs, batch_size=batch_size,
                                         validation_split=0.2, verbose=0)
                val_accuracy = max(history_temp.history['val_accuracy'])
                print(f"LR: {lr}, Units: {units}, Batch Size: {batch_size}, Epochs: {epochs}, Val Accuracy: {val_accuracy:.4f}")
                if val_accuracy > best_accuracy:
                    best_accuracy = val_accuracy
                    best_model = model
                    history = history_temp  # Save history of best model
logging.info(f"Best model: LR={lr}, Units={units}, Batch Size={batch_size}, Epochs={epochs}, Val Accuracy={best_accuracy:.4f}")

# Save best model and scaler to S3
model_path = '/content/best_model.h5'
best_model.save(model_path)
s3_client.upload_file(model_path, bucket_name, 'best_model.h5')
logging.info(f"Best model saved to S3 at s3://{bucket_name}/best_model.h5")
print("Best model saved and uploaded to S3")

scaler_path = '/content/scaler.pkl'
joblib.dump(scaler, scaler_path)
s3_client.upload_file(scaler_path, bucket_name, 'scaler.pkl')
logging.info(f"Scaler saved to S3 at s3://{bucket_name}/scaler.pkl")
print("Scaler saved and uploaded to S3")

loaded_model = best_model

X_train_std shape: (1069, 10)
Y_train shape: (1069,)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


LR: 0.001, Units: 32, Batch Size: 16, Epochs: 10, Val Accuracy: 0.5561
LR: 0.001, Units: 32, Batch Size: 16, Epochs: 20, Val Accuracy: 0.5654
LR: 0.001, Units: 32, Batch Size: 32, Epochs: 10, Val Accuracy: 0.4579
LR: 0.001, Units: 32, Batch Size: 32, Epochs: 20, Val Accuracy: 0.4953
LR: 0.001, Units: 64, Batch Size: 16, Epochs: 10, Val Accuracy: 0.5421
LR: 0.001, Units: 64, Batch Size: 16, Epochs: 20, Val Accuracy: 0.5514
LR: 0.001, Units: 64, Batch Size: 32, Epochs: 10, Val Accuracy: 0.5047
LR: 0.001, Units: 64, Batch Size: 32, Epochs: 20, Val Accuracy: 0.5607
LR: 0.01, Units: 32, Batch Size: 16, Epochs: 10, Val Accuracy: 0.5701
LR: 0.01, Units: 32, Batch Size: 16, Epochs: 20, Val Accuracy: 0.5421
LR: 0.01, Units: 32, Batch Size: 32, Epochs: 10, Val Accuracy: 0.5888
LR: 0.01, Units: 32, Batch Size: 32, Epochs: 20, Val Accuracy: 0.5607
LR: 0.01, Units: 64, Batch Size: 16, Epochs: 10, Val Accuracy: 0.5467
LR: 0.01, Units: 64, Batch Size: 16, Epochs: 20, Val Accuracy: 0.5888
LR: 0.01, Un

LR: 0.01, Units: 64, Batch Size: 32, Epochs: 20, Val Accuracy: 0.5374
Best model saved and uploaded to S3
Scaler saved and uploaded to S3


# Plot & Evaluate
1.   Evaluate model on test set.
2.   Confusion Matrix using seaborn.
Visualizes the distribution of true positives, true negatives, false positives, and false negatives.


---


* Evaluate model on test set. Assesses model performance using test data (X_test_std and Y_test) to ensure generalization.

* Plot training history. Displays accuracy and loss trends over epochs to monitor training progress and detect overfitting.

* Confusion Matrix using seaborn. Visualizes the distribution of true positives, true negatives, false positives, and false negatives for clear error analysis.

* Compute ROC AUC. Calculates the area under the ROC curve to evaluate the model’s ability to distinguish classes across thresholds.

In [7]:
# evaluate the model, plot training history, confusion matrix, and compute ROC AUC.
#Dependencies
import matplotlib.pyplot as plt #import plotting library for visualizations.
import seaborn as sns # confusion matrix.
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score # computes evaluation metrics.
import numpy as np # for array operations.

# Evaluate model on test set
y_pred = np.argmax(loaded_model.predict(X_test_std), axis=1) # converts probabilities to integer labels (0 or 1)
report = classification_report(Y_test, y_pred, output_dict=True) # generate metrics and calculates precision, recall, and F1-score.
print("Classification Report:")
print(classification_report(Y_test, y_pred))

# Confusion Matrix using seaborn
cm = confusion_matrix(Y_test, y_pred) # counts true/false positives/negatives
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive']) # plot heatmap, visualizes results with labeled axes.
plt.title("Confusion Matrix")
plt.xlabel("Predicted") # label x-axis , indicates predicted classes.
plt.ylabel("True") # label y-axis, indicates true classes.
cm_plot_path = '/content/confusion_matrix.png'
plt.savefig(cm_plot_path)
plt.close()
s3_client.upload_file(cm_plot_path, bucket_name, 'confusion_matrix.png') # upload to S3
logging.info(f"Confusion matrix saved to S3 at s3://{bucket_name}/confusion_matrix.png")
print("Confusion matrix saved and uploaded to S3")

# ROC AUC
y_pred_proba = loaded_model.predict(X_test_std)[:, 1] # get probabilities, extracts positive class probability.
roc_auc = roc_auc_score(Y_test, y_pred_proba) # calculate ROC AUC, measures classification performance.
print(f"ROC AUC Score: {roc_auc:.4f}") # display score,  show metric for review.

# Plot training history
plt.figure(figsize=(12, 4))

# Accuracy plot
plt.subplot(1, 2, 1) # define subplot,  allocates space for accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy') # plot training accuracy,  tracks learning progress.
plt.plot(history.history['val_accuracy'], label='Validation Accuracy') # plot validation accuracy,  monitors overfitting.
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1, 2, 2) # allocates space for loss.
plt.plot(history.history['loss'], label='Training Loss') # plot training loss,  tracks error reduction.
plt.plot(history.history['val_loss'], label='Validation Loss') # plot validation loss,  monitors validation error
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Save plot to S3
plot_path = '/content/training_history.png'
plt.savefig(plot_path)
plt.close()
s3_client.upload_file(plot_path, bucket_name, 'training_history.png') # upload to S3
logging.info(f"Training history plot saved to S3 at s3://{bucket_name}/training_history.png")
print("Training history plot saved and uploaded to S3")

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.46      0.46       114
           1       0.60      0.60      0.60       154

    accuracy                           0.54       268
   macro avg       0.53      0.53      0.53       268
weighted avg       0.54      0.54      0.54       268

Confusion matrix saved and uploaded to S3
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
ROC AUC Score: 0.5566
Training history plot saved and uploaded to S3


# Set up FastAPI Server with Prediction Endpoint


---


 Enables real-time predictions by deploying a (/predict/) endpoint, allowing users to upload CSV data and receive instant class predictions and probabilities, making the model practical for deployment and user interaction.

In [8]:
# Dependencies
# Packages
!pip install python-multipart  # enables file uploads in FastAPI
!pip install fastapi  # provides the Fast API framework
!pip install uvicorn  # runs the Fast API server

# libraries
import tensorflow as tf  #loading the model
import joblib  #loading the scaler
from fastapi import FastAPI, HTTPException, UploadFile, File  # API setup
from threading import Thread  #running the server in a thread
import uvicorn  #running the FastAPI server
import requests  #testing the API endpoint
import pandas as pd  #handling CSV data
import numpy as np  #array operations
import io  #processing uploaded file content
import time  #adding delay
from tensorflow.keras.optimizers import Adam  #recompiling the model
import boto3  #AWS S3 interactions
import logging  #logging actions and errors

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# AWS S3 client setup
s3_client = boto3.client('s3')
bucket_name = 'sandipproject'  # S3 bucket name.

# Load model and scaler from S3
model_path = '/content/best_model.h5'
scaler_path = '/content/scaler.pkl'
s3_client.download_file(bucket_name, 'best_model.h5', model_path)
s3_client.download_file(bucket_name, 'scaler.pkl', scaler_path)
loaded_model = tf.keras.models.load_model(model_path)

# Recompile the model to avoid the warning
loaded_model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
scaler = joblib.load(scaler_path)
logging.info("Model and scaler loaded from S3")

# Initialize FastAPI app
app = FastAPI(title="Model Prediction API")

@app.post("/predict/")
async def predict(data: UploadFile = File(...)):
    """Predict classes for uploaded CSV data."""
    try:
        file_content = await data.read()
        input_df = pd.read_csv(io.StringIO(file_content.decode('utf-8')))
        if input_df.shape[1] != 10:  # Expect 10 features, as per dataset
            raise HTTPException(status_code=400, detail=f"Expected 10 features, got {input_df.shape[1]}")
        input_std = scaler.transform(input_df)
        prediction = loaded_model.predict(input_std)
        predicted_classes = np.argmax(prediction, axis=1).tolist()
        probabilities = prediction.tolist()
        logging.info(f"Prediction made for {data.filename}: {predicted_classes}")
        return {"predictions": predicted_classes, "probabilities": probabilities}
    except Exception as e:
        logging.error(f"Prediction error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

def run_server():
    """Run Uvicorn server in a separate thread."""
    try:
        logging.info("Starting Uvicorn server on port 8000")
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info", lifespan="on")
    except Exception as e:
        logging.error(f"Server failed to start: {e}")
        print(f"Server error: {e}")

# Start server in a thread
import os
port = 8000
# Check if port is in use and free it
os.system(f"fuser -k {port}/tcp")
thread = Thread(target=run_server, daemon=True)
thread.start()
print("Server started at http://localhost:8000")
time.sleep(5)

# Test the endpoint
sample_data = X_test.iloc[:2]
sample_csv_path = '/content/sample_input.csv'
sample_data.to_csv(sample_csv_path, index=False)
files = {'data': ('sample_input.csv', open(sample_csv_path, 'rb'))}
response = requests.post('http://localhost:8000/predict/', files=files)
print("Prediction Response:", response.json())

Server started at http://localhost:8000


INFO:     Started server process [1391]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
INFO:     127.0.0.1:34798 - "POST /predict/ HTTP/1.1" 200 OK
Prediction Response: {'predictions': [0, 1], 'probabilities': [[0.527917206287384, 0.47208279371261597], [0.43361467123031616, 0.5663853287696838]]}


# DASHBOARD with Error Analysis and SHAP Explanations


---
Error Analysis
Identifies false negatives and false positives in predictions, visualizing their distribution (Feature 0 histograms), which helps pinpoint model weaknesses and guides improvements in performance or data preprocessing.



---

SHAP explanations quantify the impact of each feature on predictions, providing transparency into the neural network’s decision-making process, which is crucial for understanding and trusting the model.


---

Combines error analysis and SHAP visualizations into a single 1x2 dashboard, offering a unified view of model performance and interpretability, making it easier to present findings and analyze results holistically.

---



In [9]:
# Dependencies
# Packages
!pip install plotly  # for plotting tools
!pip install shap    # for SHAP explanations

# Libraries
import plotly.express as px  # for SHAP bar chart
import plotly.graph_objects as go  # for error histograms and other plots
from plotly.subplots import make_subplots  # for multi-plot layout
import shap  # for computing SHAP values
import pandas as pd  # for handling data
import numpy as np  # array operations
from google.colab import files  # local file download
import os  #checking file existence
import tensorflow as tf  #loading the model
import joblib  # loading the scaler
import boto3  #AWS S3 interactions
import logging  # logging actions and errors
from tensorflow.keras.optimizers import Adam  # recompiling the model
from sklearn.metrics import confusion_matrix  # computing confusion matrix

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# AWS S3 client setup
s3_client = boto3.client('s3')
bucket_name = 'sandipproject'  # S3 bucket name.

# Load model and scaler from S3
model_path = '/content/best_model.h5'
scaler_path = '/content/scaler.pkl'
s3_client.download_file(bucket_name, 'best_model.h5', model_path)
s3_client.download_file(bucket_name, 'scaler.pkl', scaler_path)
loaded_model = tf.keras.models.load_model(model_path)

# Recompile the model to match the input shape (10 features)
loaded_model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
scaler = joblib.load(scaler_path)
logging.info("Model and scaler loaded from S3")

# Verify dataset and history availability
try:
    print(f"X_test_std shape: {X_test_std.shape}")
    print(f"Y_test shape: {Y_test.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"History available: {history is not None}")
except NameError:
    # If dataset or history is not defined, create a dummy dataset and history
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    X_combined = pd.DataFrame(np.random.rand(1337, 10), columns=[f'Feature_{i}' for i in range(10)])
    y_combined = pd.Series(np.random.randint(0, 2, 1337), name='target')
    X_train, X_test, Y_train, Y_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.transform(X_test)
    # Create dummy history
    class DummyHistory:
        def __init__(self):
            self.history = {
                'accuracy': [0.5 + i * 0.03 for i in range(20)],
                'val_accuracy': [0.5 + i * 0.025 for i in range(20)],
                'loss': [0.7 - i * 0.02 for i in range(20)],
                'val_loss': [0.7 - i * 0.015 for i in range(20)]
            }
    history = DummyHistory()
    print("Dummy dataset and history created for dashboard.")
    print(f"X_test_std shape: {X_test_std.shape}")
    print(f"Y_test shape: {Y_test.shape}")
    print(f"X_test shape: {X_test.shape}")

# Error Analysis
y_pred = np.argmax(loaded_model.predict(X_test_std), axis=1)  # Predict classes
false_negatives = X_test[(Y_test == 1) & (y_pred == 0)]  # Find false negatives
false_positives = X_test[(Y_test == 0) & (y_pred == 1)]  # Find false positives

# Create subplots for error analysis
fig_error = make_subplots(rows=1, cols=2, subplot_titles=("False Negatives", "False Positives"))

# Add histograms, handle empty data
if not false_negatives.empty:
    fig_error.add_trace(go.Histogram(x=false_negatives.iloc[:, 0], name="Feature 0 (FN)", marker_color='#FF6666', opacity=0.7), row=1, col=1)
else:
    fig_error.add_trace(go.Histogram(x=[0], name="No False Negatives", marker_color='#FF6666', opacity=0.7, showlegend=False), row=1, col=1)
    fig_error.update_annotations(text="No False Negatives Found", xref="paper", yref="paper", x=0.25, y=0.5, showarrow=False, row=1, col=1)

if not false_positives.empty:
    fig_error.add_trace(go.Histogram(x=false_positives.iloc[:, 0], name="Feature 0 (FP)", marker_color='#6666FF', opacity=0.7), row=1, col=2)
else:
    fig_error.add_trace(go.Histogram(x=[0], name="No False Positives", marker_color='#6666FF', opacity=0.7, showlegend=False), row=1, col=2)
    fig_error.update_annotations(text="No False Positives Found", xref="paper", yref="paper", x=0.75, y=0.5, showarrow=False, row=1, col=2)

fig_error.update_layout(title_text="Error Analysis: Distribution of Feature 0", bargap=0.2)

# Save Error Analysis to S3
error_local_file = '/content/error_analysis.html'
fig_error.write_html(error_local_file)
s3_client.upload_file(error_local_file, bucket_name, 'error_analysis.html')
logging.info(f"Error analysis saved to S3 at s3://{bucket_name}/error_analysis.html")
print(f"Error analysis uploaded to S3 at s3://{bucket_name}/error_analysis.html")

# SHAP Explanations
background_data = shap.sample(X_test_std, 10)  # Sample background data
test_data = X_test_std[:10]  # Select test subset
explainer = shap.KernelExplainer(lambda x: loaded_model.predict(x)[:, 1], background_data)
shap_values = explainer.shap_values(test_data)  # Compute SHAP values

# Define meaningful feature names for the Diabetes dataset with 10 features
feature_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Feature_Extra1",
    "Feature_Extra2"
]
shap_df = pd.DataFrame(shap_values, columns=feature_names)

# Create SHAP bar plot with meaningful feature names
shap_mean = shap_df.abs().mean()
fig_shap = go.Figure()
fig_shap.add_trace(go.Bar(
    x=shap_mean.values,
    y=shap_mean.index,
    orientation='h',
    marker_color='#800080',  # Darker purple for consistency
    name='SHAP Values'
))
fig_shap.update_layout(
    title="SHAP Feature Importance (Class 1)",
    xaxis_title="Mean |SHAP Value|",
    yaxis_title="Feature",
    height=400,
    width=600
)

# Save SHAP Explanations to S3
shap_local_file = '/content/shap_explanations.html'
fig_shap.write_html(shap_local_file)
s3_client.upload_file(shap_local_file, bucket_name, 'shap_explanations.html')
logging.info(f"SHAP explanations saved to S3 at s3://{bucket_name}/shap_explanations.html")
print(f"SHAP explanations uploaded to S3 at s3://{bucket_name}/shap_explanations.html")

# Compute Confusion Matrix as a Plotly Heatmap
cm = confusion_matrix(Y_test, y_pred)
fig_cm = go.Figure(data=go.Heatmap(
    z=cm,
    x=['Negative', 'Positive'],
    y=['Negative', 'Positive'],
    text=cm,
    texttemplate="%{text}",
    colorscale='Blues',
    showscale=False
))
fig_cm.update_layout(
    title="Confusion Matrix",
    xaxis_title="Predicted",
    yaxis_title="True",
    height=400,
    width=400
)

# Create Training History Plot with Plotly
fig_history = make_subplots(rows=1, cols=2, subplot_titles=("Model Accuracy", "Model Loss"))
# Accuracy
fig_history.add_trace(go.Scatter(
    x=list(range(1, len(history.history['accuracy']) + 1)),
    y=history.history['accuracy'],
    mode='lines',
    name='Training Accuracy',
    line=dict(color='#00CC00')  # Green
), row=1, col=1)
fig_history.add_trace(go.Scatter(
    x=list(range(1, len(history.history['val_accuracy']) + 1)),
    y=history.history['val_accuracy'],
    mode='lines',
    name='Validation Accuracy',
    line=dict(color='#FF9900')  # Orange
), row=1, col=1)
# Loss
fig_history.add_trace(go.Scatter(
    x=list(range(1, len(history.history['loss']) + 1)),
    y=history.history['loss'],
    mode='lines',
    name='Training Loss',
    line=dict(color='#00CC00')
), row=1, col=2)
fig_history.add_trace(go.Scatter(
    x=list(range(1, len(history.history['val_loss']) + 1)),
    y=history.history['val_loss'],
    mode='lines',
    name='Validation Loss',
    line=dict(color='#FF9900')
), row=1, col=2)
fig_history.update_layout(
    height=400,
    width=800,
    showlegend=True
)
fig_history.update_xaxes(title_text="Epoch", row=1, col=1)
fig_history.update_xaxes(title_text="Epoch", row=1, col=2)
fig_history.update_yaxes(title_text="Accuracy", row=1, col=1)
fig_history.update_yaxes(title_text="Loss", row=1, col=2)

# Combine into Dashboard with Four Rows
dashboard = make_subplots(
    rows=4, cols=1,  # Four rows, one column
    subplot_titles=("Error Analysis", "SHAP Feature Importance", "Confusion Matrix", "Training History"),
    row_heights=[0.25, 0.25, 0.25, 0.25],  # Equal height for all rows
    vertical_spacing=0.1
)

# Row 1: Error Analysis (False Negatives and False Positives side by side)
error_subplot = make_subplots(rows=1, cols=2, subplot_titles=("False Negatives", "False Positives"))
if not false_negatives.empty:
    error_subplot.add_trace(go.Histogram(x=false_negatives.iloc[:, 0], name="Feature 0 (FN)", marker_color='#FF6666', opacity=0.7), row=1, col=1)
else:
    error_subplot.add_trace(go.Histogram(x=[0], name="No False Negatives", marker_color='#FF6666', opacity=0.7, showlegend=False), row=1, col=1)
    error_subplot.update_annotations(text="No False Negatives Found", xref="paper", yref="paper", x=0.25, y=0.5, showarrow=False, row=1, col=1)

if not false_positives.empty:
    error_subplot.add_trace(go.Histogram(x=false_positives.iloc[:, 0], name="Feature 0 (FP)", marker_color='#6666FF', opacity=0.7), row=1, col=2)
else:
    error_subplot.add_trace(go.Histogram(x=[0], name="No False Positives", marker_color='#6666FF', opacity=0.7, showlegend=False), row=1, col=2)
    error_subplot.update_annotations(text="No False Positives Found", xref="paper", yref="paper", x=0.75, y=0.5, showarrow=False, row=1, col=2)

for trace in error_subplot.data:
    trace.showlegend = True
    dashboard.add_trace(trace, row=1, col=1)

# Row 2: SHAP Feature Importance (spanning full width)
dashboard.add_trace(
    go.Bar(
        x=shap_mean.values,
        y=shap_mean.index,  # Now uses meaningful feature names
        orientation='h',
        marker_color='#800080',
        name='SHAP Values'
    ),
    row=2, col=1
)

# Row 3: Confusion Matrix (spanning full width)
for trace in fig_cm.data:
    dashboard.add_trace(trace, row=3, col=1)

# Row 4: Training History (spanning full width)
for trace in fig_history.data:
    dashboard.add_trace(trace, row=4, col=1)

# Update layout with styling
dashboard.update_layout(
    height=1600,  # Increased height for four rows
    width=1200,
    title_text="Model Insights Dashboard: Comprehensive Analysis",
    title_font=dict(size=24, color='#333333', family='Arial'),
    bargap=0.2,
    showlegend=True,
    margin=dict(t=150),  # Space for summary text
    plot_bgcolor='#F5F5F5',  # Light gray background
    paper_bgcolor='#FFFFFF'  # White paper background
)

# Add Summary as an annotation above the visualizations
summary_text = (
    "Dashboard shows ML model analysis:Error Analysis (misclassifications), + SHAP (feature impact), + Confusion Matrix (prediction performance), + Training History (learning over epochs)."
)
dashboard.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.05,
    showarrow=False,
    font=dict(size=14, color='#333333', family='Arial'),
    align='center',
    bordercolor='#CCCCCC',
    borderwidth=1,
    borderpad=10,
    bgcolor='#E6E6E6',
    opacity=0.9
)

# Save Dashboard to S3 and Download Locally
local_file = '/content/dashboard.html'
try:
    dashboard.write_html(local_file)
    if os.path.exists(local_file):
        print(f"Dashboard created locally: {local_file}")
    else:
        raise FileNotFoundError(f"Failed to create dashboard: {local_file}")

    s3_key = 'dashboard.html'
    s3_client.upload_file(local_file, bucket_name, s3_key)
    logging.info(f"Dashboard saved to S3 at s3://{bucket_name}/{s3_key}")
    print(f"Dashboard uploaded to S3 at s3://{bucket_name}/{s3_key}")

    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=s3_key)
    if 'Contents' in response and any(obj['Key'] == s3_key for obj in response['Contents']):
        print(f"Verified: {s3_key} exists in S3")
    else:
        print(f"Warning: {s3_key} not found in S3 after upload")

    files.download(local_file)
except Exception as e:
    logging.error(f"Error in dashboard creation/upload: {e}")
    print(f"Error: {e}")

X_test_std shape: (268, 10)
Y_test shape: (268,)
X_test shape: (268, 10)
History available: True
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
Error analysis uploaded to S3 at s3://sandipproject/error_analysis.html
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step


  0%|          | 0/10 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
SHAP explanations uploaded to S3 at s3://sandipproject/shap_explanations.html
Dashboard created locally: /content/dashboard.html
Dashboard uploaded to S3 at s3://sandipproject/dashboard.html
Verified: dashboard

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>